# Final Project — Power Consumption (Tetouan)
### Michelle Silveira

This notebook fits an **elastic net regression** on the Power Consumption data
with PySpark MLlib, then uses the fitted pipeline to score a **structured
stream** of incoming CSV files.

It is written to run unchanged in **Google Colab** *and* in a **JupyterHub /
local Jupyter** environment.

**Workflow**

1. Set up Spark (and install it on Colab if needed).
2. Load `power_ml_data.csv` into a pandas DataFrame, then to a Spark
   DataFrame.
3. Build a pipeline: SQL rename + cast → Binarizer (Hour) → OneHotEncoder
   (Month) → VectorAssembler + PCA (weather) → final VectorAssembler →
   `LinearRegression`.
4. Fit with 5-fold `CrossValidator` over an 11 × 11 grid of `regParam` and
   `elasticNetParam`. Report the best params, CV RMSE and training RMSE.
5. Read a stream of CSV files, score each batch with the fitted pipeline,
   compute residuals, join two views of the stream on `label`, and write the
   result to the console.

> Pair this notebook with `produce_data.py` — that script drops sampled CSV
> files into the watched folder while the streaming query runs.

## 1. Environment setup
This code is to adjust the case where I am running on Colab or JupytherHUb from NC State
Installs PySpark only if it isn't already importable. On a JupyterHub that already has Spark this is a no-op.

In [3]:
# Install PySpark only when missing (Colab needs it; most JupyterHubs already have it).
import importlib, subprocess, sys
# Spark + Python imports used throughout the notebook.
import os
import time
from pathlib import Path

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, DoubleType, IntegerType,
)

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    SQLTransformer, Binarizer, OneHotEncoder,
    VectorAssembler, PCA,
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

#forcing an output to make sure code is running
print("Imports completed successfully")


Imports completed successfully


In [4]:
# Build (or reuse) a SparkSession. ``getOrCreate`` is safe in both Colab and JupyterHub.
spark = (
    SparkSession.builder
    .appName("PowerConsumptionFinalProject")
    .config("spark.sql.shuffle.partitions", "8")  # small data — keep shuffles lean
    .getOrCreate()
)

# Quieter logs make the streaming output readable.
spark.sparkContext.setLogLevel("WARN")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 21:18:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/29 21:18:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/29 21:18:05 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


## 2. Load the modelling data

Reads `power_ml_data.csv` into pandas (per the rubric), then converts to a Spark DataFrame. I try a local path first and fall back to the published URL so the notebook is portable in case we would be receiving from the ncstate url given by the Project description

In [5]:
# Read the local modelling CSV from the same folder as this notebook
ML_LOCAL = "power_ml_data.csv"

if not Path(ML_LOCAL).is_file():
    raise FileNotFoundError(
        f"Could not find {ML_LOCAL}. warning to check if i upload the file given in moodel locally."
    )

print(f"Reading modelling data from local file: {ML_LOCAL}")

# Read with pandas as the assignment requires
pandas_df = pd.read_csv(ML_LOCAL)

print("pandas shape:", pandas_df.shape)
pandas_df.head()

Reading modelling data from local file: power_ml_data.csv
pandas shape: (47174, 10)


,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


In [6]:
# Convert the pandas frame to a Spark DataFrame.
spark_df = spark.createDataFrame(pandas_df)
spark_df.printSchema()
spark_df.show(5)
print("rows:", spark_df.count())

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

rows: 47174


## 3. Build the modelling pipeline

Each step is an MLlib component so the whole thing fits inside a `Pipeline`
that we can reuse on the streaming side.

| # | Transformer                                                                 | Purpose                                                              |
|---|-----------------------------------------------------------------------------|----------------------------------------------------------------------|
| 1 | `SQLTransformer` — `Power_Zone_3 AS label` and `CAST(Hour AS DOUBLE)`       | Rename the response and ensure `Hour` is a `DoubleType`.             |
| 2 | `Binarizer` (threshold = 6.5)                                                | Converts `Hour` into a night/day flag.                               |
| 3 | `OneHotEncoder` (Month)                                                      | One-hot encodes the month indicator.                                 |
| 4 | `VectorAssembler` + `PCA` (k=2)                                              | Two principal components on the weather columns.                     |
| 5 | Final `VectorAssembler`                                                      | Builds the `features` vector used by Linear Regression.              |
| 6 | `LinearRegression` (elastic net)                                             | The estimator we tune via cross-validation.                          |


In [7]:
# Step 1: rename Power_Zone_3 -> label AND cast Hour to DoubleType in one SQLTransformer.
# Putting the rename inside the pipeline means the same pipeline transforms the
# streaming data later without us having to remember to rename anything.
sql_prep = SQLTransformer(statement="""
    SELECT
        Temperature, Humidity, Wind_Speed,
        General_Diffuse_Flows, Diffuse_Flows,
        Power_Zone_1, Power_Zone_2,
        Month,
        CAST(Hour AS DOUBLE) AS Hour,
        Power_Zone_3 AS label
    FROM __THIS__
""")

In [9]:
# Step 2: Binarize the Hour column at 6.5 (night vs. day).
hour_binarizer = Binarizer(
    threshold=6.5, inputCol="Hour", outputCol="Hour_binary",
)

# Step 3: One-hot encode Month. Month is already integer-valued so we can feed
# it straight into OneHotEncoder without a StringIndexer.
month_ohe = OneHotEncoder(
    inputCol="Month", outputCol="Month_ohe",
)

In [10]:
# Step 4: PCA on the weather columns. We first assemble them into a vector,
# then fit a 2-component PCA on top.
weather_cols = [
    "Temperature", "Humidity", "Wind_Speed",
    "General_Diffuse_Flows", "Diffuse_Flows",
]

weather_assembler = VectorAssembler(
    inputCols=weather_cols, outputCol="weather_vec",
)

pca = PCA(k=2, inputCol="weather_vec", outputCol="weather_pca")


In [11]:
# Step 5: assemble the final feature vector consumed by LinearRegression.
final_assembler = VectorAssembler(
    inputCols=[
        "weather_pca",   # two PCA components
        "Hour_binary",   # day/night flag
        "Power_Zone_1",
        "Power_Zone_2",
        "Month_ohe",     # one-hot Month
    ],
    outputCol="features",
)

# Step 6: the elastic net estimator. CrossValidator will tune regParam and
# elasticNetParam. featuresCol/labelCol must match what the pipeline produces.
lr = LinearRegression(featuresCol="features", labelCol="label")

# Glue everything together.
pipeline = Pipeline(stages=[
    sql_prep,
    hour_binarizer,
    month_ohe,
    weather_assembler,
    pca,
    final_assembler,
    lr,
])


## 4. Cross-validated elastic net

11 × 11 = 121 candidate models, each evaluated with 5-fold CV on RMSE. This is
the longest cell in the notebook — feel free to shrink the grid while
experimenting.

In [12]:
# 11 values for each tuning parameter, exactly as specified in the rubric.
reg_values = [0.0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1.0]
enet_values = [0.0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1.0]

param_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, reg_values)
    .addGrid(lr.elasticNetParam, enet_values)
    .build()
)

print(f"grid size: {len(param_grid)}  (= {len(reg_values)} x {len(enet_values)})")

grid size: 121  (= 11 x 11)


In [13]:
# 5-fold cross-validation, optimising RMSE. 
rmse_evaluator = RegressionEvaluator(
    labelCol="label", predictionCol="prediction", metricName="rmse",
)

cross_val = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=rmse_evaluator,
    numFolds=5,
    parallelism=2,   # fit two folds at a time when resources allow
)

# Fit. This is the slow cell. I am generating the time in seconds in google colab where i developed this it took 757.3 seconds
start = time.time()
cv_model = cross_val.fit(spark_df)
print(f"CV fit took {time.time() - start:,.1f} seconds")

26/04/29 21:27:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/29 21:27:39 WARN Instrumentation: [b1fc6375] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 21:27:39 WARN Instrumentation: [db8e8079] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 21:27:40 WARN Instrumentation: [b1fc6375] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/29 21:27:40 WARN Instrumentation: [db8e8079] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/29 21:27:42 WARN Instrumentation: [e4af3a34] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 21:27:42 WARN Instrumentation: [f14f66e5] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 21:27:42 W

CV fit took 342.0 seconds


## 5. Report the chosen model


In [14]:
# The fitted Pipeline (with the LR stage at the end) is the best estimator.
best_pipeline_model = cv_model.bestModel
best_lr_model = best_pipeline_model.stages[-1]

best_reg = best_lr_model.getRegParam()
best_enet = best_lr_model.getElasticNetParam()

# avgMetrics is parallel to estimatorParamMaps; the smallest entry wins for RMSE.
best_index = min(range(len(cv_model.avgMetrics)), key=cv_model.avgMetrics.__getitem__)
cv_rmse = cv_model.avgMetrics[best_index]

print("Optimal hyper-parameters")
print(f"  regParam        = {best_reg}")
print(f"  elasticNetParam = {best_enet}")
print(f"Cross-validated RMSE: {cv_rmse:,.4f}")

Optimal hyper-parameters
  regParam        = 0.1
  elasticNetParam = 0.05
Cross-validated RMSE: 2,147.8215


In [15]:
# Training-set RMSE: use the fitted pipeline as a transformer, then evaluate.
train_predictions = best_pipeline_model.transform(spark_df)
train_rmse = rmse_evaluator.evaluate(train_predictions)
print(f"Training-set RMSE: {train_rmse:,.4f}")


Training-set RMSE: 2,147.0973


In [16]:
# Add a residual column = label - prediction and show it next to the predictions.
train_with_residuals = train_predictions.withColumn(
    "residual", F.col("label") - F.col("prediction"),
)

train_with_residuals.select("label", "prediction", "residual").show(20, truncate=False)

+-----------+------------------+------------------+
|label      |prediction        |residual          |
+-----------+------------------+------------------+
|20240.96386|20878.89469689072 |-637.9308368907186|
|20131.08434|18660.13994797041 |1470.9443920295926|
|19668.43373|18204.665170716016|1463.7685592839844|
|18899.27711|17590.565897218486|1308.7112127815126|
|18442.40964|16997.220046537903|1445.1895934620989|
|18130.12048|16517.6069315177  |1612.5135484823004|
|17945.06024|16093.16876131072 |1851.8914786892783|
|17459.27711|15722.61910167929 |1736.6580083207082|
|17025.54217|15270.969724501367|1754.572445498634 |
|16794.21687|14938.275405728531|1855.9414642714692|
|16638.07229|14652.31774788132 |1985.7545421186805|
|16395.18072|14414.835672219506|1980.3450477804945|
|16117.59036|14082.818590625706|2034.7717693742943|
|15822.6506 |13624.814099621348|2197.836500378653 |
|15672.28916|13450.276790086835|2222.0123699131655|
|15597.10843|13302.187722922197|2294.920707077803 |
|15510.36145

## 6. Streaming part

Pair this notebook with `produce_data.py`, which writes 5-row CSV samples into
the folder below every 10 seconds.

We do **two transformations on the same stream**, then join them on `label`:

1. Score the stream with `best_pipeline_model`, attach the residual, and keep
   only `label, prediction, residual`.
2. Take the original stream and rename `Power_Zone_3` to `label`.

The joined frame is written to the console with `outputMode="append"`.

In [17]:
# Watched folder — make sure it exists. Same default as produce_data.py. code that generates the stream files
STREAM_DIR = Path("stream_data").resolve()
STREAM_DIR.mkdir(parents=True, exist_ok=True)
print("watching:", STREAM_DIR)


watching: /home/jupyter-masilvei@ncsu.edu/stream_data


In [18]:
# Schema must match the columns produced by produce_data.py (same as power_ml_data.csv).
stream_schema = StructType([
    StructField("Temperature",             DoubleType(),  True),
    StructField("Humidity",                DoubleType(),  True),
    StructField("Wind_Speed",              DoubleType(),  True),
    StructField("General_Diffuse_Flows",   DoubleType(),  True),
    StructField("Diffuse_Flows",           DoubleType(),  True),
    StructField("Power_Zone_1",            DoubleType(),  True),
    StructField("Power_Zone_2",            DoubleType(),  True),
    StructField("Power_Zone_3",            DoubleType(),  True),
    StructField("Month",                   IntegerType(), True),
    StructField("Hour",                    IntegerType(), True),
])

# Read the stream of CSV files. header=True skips the column line in each file.
stream_df = (
    spark.readStream
    .schema(stream_schema)
    .option("header", True)
    .option("maxFilesPerTrigger", 1)   # one file per micro-batch keeps output tidy
    .csv(str(STREAM_DIR))
)

print("stream isStreaming:", stream_df.isStreaming)

stream isStreaming: True


In [19]:
# Transformation A: predictions + residual from the fitted pipeline.
# The pipeline already renames Power_Zone_3 -> label, so we get label out the
# other end and can subtract the prediction directly.
predictions_stream = (
    best_pipeline_model
    .transform(stream_df)
    .withColumn("residual", F.col("label") - F.col("prediction"))
    .select("label", "prediction", "residual")
)

# Transformation B: same stream, just rename Power_Zone_3 to label.
renamed_stream = stream_df.withColumnRenamed("Power_Zone_3", "label")

# Join the two transformations of the same stream on the common ``label`` column.
joined_stream = predictions_stream.join(renamed_stream, on="label")


In [21]:
# Write to the console in append mode and start the query.
query = (
    joined_stream.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", False)
    .option("numRows", 20)
    .option("checkpointLocation", "checkpoint_stream")
    .start()
)

26/04/29 21:44:59 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


### Run the producer ( the .py code requested in the Project Requeriment -> Final.pdf)

Once the streaming query is running, the cell below feeds it data by
shelling out to `produce_data.py`. It writes 20 small CSV files into
`stream_data/` with a 10-second pause between each — about 200
seconds total. While it runs, scroll back up to the previous cell's
output to watch the streaming predictions arrive in real time.

**I run this code only if the previous query is already run**

In [22]:
# Run the producer (20 iterations x 5 rows, 10s apart). This blocks the
# cell for ~200 seconds while it writes files into stream_data/.
# Watch the output of the previous cell to see the streaming predictions.
!python produce_data.py

[producer] reading source data from /home/jupyter-masilvei@ncsu.edu/power_streaming_data.csv
[producer] loaded 5,242 rows; columns = ['Temperature', 'Humidity', 'Wind_Speed', 'General_Diffuse_Flows', 'Diffuse_Flows', 'Power_Zone_1', 'Power_Zone_2', 'Power_Zone_3', 'Month', 'Hour']
[producer] writing 20 files of 5 rows to /home/jupyter-masilvei@ncsu.edu/stream_data
[producer] pausing 10s between writes

[producer] (01/20) wrote 5 rows -> stream_001_20260429_214617_651476.csv
-------------------------------------------
Batch: 0
-------------------------------------------
-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+----------

[producer] (02/20) wrote 5 rows -> stream_002_20260429_214627_666450.csv
-------------------------------------------
Batch: 1
-------------------------------------------
-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|label      |prediction        |residual           |Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|10978.71297|11412.813017916229|-434.1000479162285 |23.89      |40.68   |4.923     |236.3                |124.4        |29341.59292 |17898.9605  |9    |8   |
|18731.56627|13634.229780026457|5097.3364899735425 |20.95      |61.77   |0.068     |525.7            